# CSC4093/DSC4213: Neural Networks and Deep Learning
## Programming Assignment 01 — LSTMs
### Personal Health Mention Classification from Tweets
---
**Task:** Binary classification — Personal Health Mention `(1)` vs Non-Personal `(0)`  
**Models:** LSTM and Bidirectional LSTM (Bi-LSTM)

In [ ]:
import sys
!{sys.executable} -m pip install tensorflow nltk scikit-learn matplotlib seaborn pandas numpy --quiet
print('All packages ready.')

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from nltk.corpus import stopwords
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional, SpatialDropout1D
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam

nltk.download('stopwords')
english_stops = set(stopwords.words('english'))

# Set seeds for reproducibility
import random
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

print('TensorFlow version:', tf.__version__)
print('All libraries imported.')

## Step 2: Load and Preview Dataset

In [ ]:
train_data = pd.read_csv('phm_train.csv')
test_data  = pd.read_csv('phm_test.csv')

print('Training set shape:', train_data.shape)
print('Test set shape    :', test_data.shape)
print()
print('Sample training rows:')
print(train_data[['label','tweet']].head(5))
print()
print('Label distribution — Training:')
vc = train_data['label'].value_counts()
print(f'  Class 0 (Non-Personal Health): {vc[0]} ({vc[0]/len(train_data)*100:.1f}%)')
print(f'  Class 1 (Personal Health)    : {vc[1]} ({vc[1]/len(train_data)*100:.1f}%)')
print()
print('Label distribution — Test:')
vct = test_data['label'].value_counts()
print(f'  Class 0 (Non-Personal Health): {vct[0]} ({vct[0]/len(test_data)*100:.1f}%)')
print(f'  Class 1 (Personal Health)    : {vct[1]} ({vct[1]/len(test_data)*100:.1f}%)')

## Step 3: Text Preprocessing

In [ ]:
def preprocess_tweets(data):
    x_data = data['tweet'].copy().astype(str)
    y_data = data['label'].copy()

    # Clean text
    x_data = x_data.str.replace(r'http\S+|www\S+', '', regex=True)   # remove URLs
    x_data = x_data.str.replace(r'user_mention', '', regex=True)       # remove mention placeholder
    x_data = x_data.str.replace(r'emo_\w+', '', regex=True)           # remove emotion tags
    x_data = x_data.str.replace(r'<.*?>', '', regex=True)             # remove HTML
    x_data = x_data.str.replace(r'[^A-Za-z\s]', ' ', regex=True)     # keep only letters
    x_data = x_data.str.lower().str.strip()

    # Tokenize and remove stopwords — keep words of length > 1
    x_data = x_data.apply(lambda t: [
        w for w in t.split()
        if w not in english_stops and len(w) > 1
    ])
    return x_data, y_data

x_train, y_train = preprocess_tweets(train_data)
x_test,  y_test  = preprocess_tweets(test_data)

print('Preprocessing complete.')
print()
print('Example tweet (raw)      :', train_data['tweet'].iloc[2])
print('Example tweet (processed):', x_train.iloc[2])
print('Label                    :', y_train.iloc[2])

## Step 4: Tokenization and Padding

In [ ]:
# Use mean length — consistent with sample notebook approach
def get_max_length(x_data):
    return int(np.ceil(np.mean([len(t) for t in x_data])))

# Fit tokenizer only on training data
token = Tokenizer(lower=False)   # already lowered in preprocessing
token.fit_on_texts(x_train)

x_train_seq = token.texts_to_sequences(x_train)
x_test_seq  = token.texts_to_sequences(x_test)

max_length  = get_max_length(x_train)
total_words = len(token.word_index) + 1

x_train_pad = pad_sequences(x_train_seq, maxlen=max_length, padding='post', truncating='post')
x_test_pad  = pad_sequences(x_test_seq,  maxlen=max_length, padding='post', truncating='post')

print('Vocabulary size      :', total_words)
print('Max tweet length     :', max_length)
print('Encoded X Train shape:', x_train_pad.shape)
print('Encoded X Test  shape:', x_test_pad.shape)
print()
print('Encoded X Train (first 3 rows):')
print(x_train_pad[:3])

## Step 5: Compute Class Weights

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
cw = dict(enumerate(class_weights))
print('Class weights:')
print(f'  Class 0 (Non-Personal): {cw[0]:.4f}')
print(f'  Class 1 (Personal)    : {cw[1]:.4f}')

## Step 6: Hyperparameters

In [ ]:
# ── Hyperparameters ──────────────────────────────────────────────────
EMBED_DIM      = 64     # Embedding size — matches sample notebook baseline
LSTM_UNITS     = 64     # Single LSTM layer units
SPATIAL_DROP   = 0.2    # SpatialDropout1D after embedding layer
RECURRENT_DROP = 0.2    # Dropout inside LSTM recurrent connections
DENSE_DROP     = 0.5    # Dropout before output Dense layer
BATCH_SIZE     = 128    # Matches sample notebook
EPOCHS         = 10     # EarlyStopping will stop earlier if needed
LEARNING_RATE  = 0.001
# ────────────────────────────────────────────────────────────────────
import os
os.makedirs('models', exist_ok=True)
print('Hyperparameters set.')

## Step 7: Build Model 1 — LSTM

In [ ]:
lstm_model = Sequential(name='LSTM_Model')
lstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
lstm_model.add(SpatialDropout1D(SPATIAL_DROP))
lstm_model.add(LSTM(
    LSTM_UNITS,
    dropout=RECURRENT_DROP,
    recurrent_dropout=RECURRENT_DROP
))
lstm_model.add(Dropout(DENSE_DROP))
lstm_model.add(Dense(1, activation='sigmoid'))

lstm_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(lstm_model.summary())

## Step 8: Train LSTM Model

In [ ]:
lstm_callbacks = [
    ModelCheckpoint(
        'models/LSTM_PHM.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

lstm_history = lstm_model.fit(
    x_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    class_weight=cw,
    callbacks=lstm_callbacks,
    verbose=1
)

## Step 9: Build Model 2 — Bi-LSTM

In [ ]:
bilstm_model = Sequential(name='BiLSTM_Model')
bilstm_model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
bilstm_model.add(SpatialDropout1D(SPATIAL_DROP))
bilstm_model.add(Bidirectional(LSTM(
    LSTM_UNITS,
    dropout=RECURRENT_DROP,
    recurrent_dropout=RECURRENT_DROP
)))
bilstm_model.add(Dropout(DENSE_DROP))
bilstm_model.add(Dense(1, activation='sigmoid'))

bilstm_model.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='binary_crossentropy',
    metrics=['accuracy']
)
print(bilstm_model.summary())

## Step 10: Train Bi-LSTM Model

In [ ]:
bilstm_callbacks = [
    ModelCheckpoint(
        'models/BiLSTM_PHM.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    ),
    EarlyStopping(
        monitor='val_accuracy',
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

bilstm_history = bilstm_model.fit(
    x_train_pad, y_train,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_split=0.1,
    class_weight=cw,
    callbacks=bilstm_callbacks,
    verbose=1
)

## Step 11: Evaluate Both Models on Test Set

In [ ]:
# LSTM predictions
lstm_probs = lstm_model.predict(x_test_pad)
lstm_pred  = (lstm_probs >= 0.5).astype(int).flatten()
lstm_correct = int(np.sum(lstm_pred == np.array(y_test)))
lstm_acc     = lstm_correct / len(lstm_pred) * 100

# Bi-LSTM predictions
bilstm_probs = bilstm_model.predict(x_test_pad)
bilstm_pred  = (bilstm_probs >= 0.5).astype(int).flatten()
bilstm_correct = int(np.sum(bilstm_pred == np.array(y_test)))
bilstm_acc     = bilstm_correct / len(bilstm_pred) * 100

print('=' * 55)
print('           LSTM — Test Results')
print('=' * 55)
print(f'Correct Predictions : {lstm_correct}')
print(f'Wrong Predictions   : {len(lstm_pred) - lstm_correct}')
print(f'Test Accuracy       : {lstm_acc:.2f}%')
print()
print('Classification Report (LSTM):')
print(classification_report(
    y_test, lstm_pred,
    target_names=['Non-Personal (0)', 'Personal Health (1)']
))

print('=' * 55)
print('         Bi-LSTM — Test Results')
print('=' * 55)
print(f'Correct Predictions : {bilstm_correct}')
print(f'Wrong Predictions   : {len(bilstm_pred) - bilstm_correct}')
print(f'Test Accuracy       : {bilstm_acc:.2f}%')
print()
print('Classification Report (Bi-LSTM):')
print(classification_report(
    y_test, bilstm_pred,
    target_names=['Non-Personal (0)', 'Personal Health (1)']
))

## Step 12: Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Confusion Matrices — LSTM vs Bi-LSTM', fontsize=14, fontweight='bold')
labels = ['Non-Personal (0)', 'Personal Health (1)']

cm_lstm   = confusion_matrix(y_test, lstm_pred)
cm_bilstm = confusion_matrix(y_test, bilstm_pred)

sns.heatmap(cm_lstm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0],
            linewidths=0.5)
axes[0].set_title(f'LSTM  (Test Accuracy: {lstm_acc:.2f}%)')
axes[0].set_ylabel('Actual Label')
axes[0].set_xlabel('Predicted Label')
axes[0].tick_params(axis='x', rotation=15)

sns.heatmap(cm_bilstm, annot=True, fmt='d', cmap='Greens',
            xticklabels=labels, yticklabels=labels, ax=axes[1],
            linewidths=0.5)
axes[1].set_title(f'Bi-LSTM  (Test Accuracy: {bilstm_acc:.2f}%)')
axes[1].set_ylabel('Actual Label')
axes[1].set_xlabel('Predicted Label')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: confusion_matrices.png')

## Step 13: Accuracy and Loss Plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('LSTM vs Bi-LSTM — Training Performance', fontsize=15, fontweight='bold')

axes[0,0].plot(lstm_history.history['accuracy'],     label='Train', color='royalblue',  lw=2)
axes[0,0].plot(lstm_history.history['val_accuracy'], label='Val',   color='darkorange', lw=2)
axes[0,0].set_title('LSTM — Accuracy'); axes[0,0].set_xlabel('Epoch'); axes[0,0].set_ylabel('Accuracy')
axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)

axes[0,1].plot(lstm_history.history['loss'],     label='Train', color='royalblue',  lw=2)
axes[0,1].plot(lstm_history.history['val_loss'], label='Val',   color='darkorange', lw=2)
axes[0,1].set_title('LSTM — Loss'); axes[0,1].set_xlabel('Epoch'); axes[0,1].set_ylabel('Loss')
axes[0,1].legend(); axes[0,1].grid(True, alpha=0.3)

axes[1,0].plot(bilstm_history.history['accuracy'],     label='Train', color='seagreen', lw=2)
axes[1,0].plot(bilstm_history.history['val_accuracy'], label='Val',   color='crimson',  lw=2)
axes[1,0].set_title('Bi-LSTM — Accuracy'); axes[1,0].set_xlabel('Epoch'); axes[1,0].set_ylabel('Accuracy')
axes[1,0].legend(); axes[1,0].grid(True, alpha=0.3)

axes[1,1].plot(bilstm_history.history['loss'],     label='Train', color='seagreen', lw=2)
axes[1,1].plot(bilstm_history.history['val_loss'], label='Val',   color='crimson',  lw=2)
axes[1,1].set_title('Bi-LSTM — Loss'); axes[1,1].set_xlabel('Epoch'); axes[1,1].set_ylabel('Loss')
axes[1,1].legend(); axes[1,1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('accuracy_loss_plots.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: accuracy_loss_plots.png')

## Step 14: Performance Comparison Table

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

comparison = pd.DataFrame({
    'Metric': [
        'Test Accuracy (%)',
        'Precision (weighted %)',
        'Recall (weighted %)',
        'F1-Score (weighted %)',
        'Best Val Accuracy (%)',
        'Best Val Loss'
    ],
    'LSTM': [
        f'{lstm_acc:.2f}',
        f'{precision_score(y_test, lstm_pred,   average="weighted")*100:.2f}',
        f'{recall_score(y_test,    lstm_pred,   average="weighted")*100:.2f}',
        f'{f1_score(y_test,        lstm_pred,   average="weighted")*100:.2f}',
        f"{max(lstm_history.history['val_accuracy'])*100:.2f}",
        f"{min(lstm_history.history['val_loss']):.4f}"
    ],
    'Bi-LSTM': [
        f'{bilstm_acc:.2f}',
        f'{precision_score(y_test, bilstm_pred, average="weighted")*100:.2f}',
        f'{recall_score(y_test,    bilstm_pred, average="weighted")*100:.2f}',
        f'{f1_score(y_test,        bilstm_pred, average="weighted")*100:.2f}',
        f"{max(bilstm_history.history['val_accuracy'])*100:.2f}",
        f"{min(bilstm_history.history['val_loss']):.4f}"
    ]
})

print('=' * 60)
print('          Model Performance Comparison')
print('=' * 60)
print(comparison.to_string(index=False))
print('=' * 60)

## Step 15: Discussion

### Model Architectures

Both models use an identical single-layer architecture to ensure a fair comparison, differing only in whether the recurrent layer is unidirectional or bidirectional:

```
Embedding(vocab_size, 64, input_length=max_length)
    ↓
SpatialDropout1D(0.2)
    ↓
LSTM(64) / Bidirectional(LSTM(64))  ← only difference
[dropout=0.2, recurrent_dropout=0.2]
    ↓
Dropout(0.5)
    ↓
Dense(1, activation='sigmoid')
```

### Hyperparameter Choices

| Hyperparameter | Value | Rationale |
|---|---|---|
| Embedding dim | 64 | Balanced representation for tweet vocabulary |
| LSTM units | 64 | Sufficient capacity without overfitting short tweets |
| SpatialDropout1D | 0.2 | Drops entire embedding channels — more effective than standard dropout for NLP |
| Recurrent dropout | 0.2 | Regularizes hidden-to-hidden connections inside LSTM |
| Dense dropout | 0.5 | Strong regularization before final prediction layer |
| Batch size | 128 | Stable gradient estimates for this dataset size |
| ReduceLROnPlateau | factor=0.5, patience=2 | Halves LR when val_loss plateaus |
| Class weights | Balanced | Corrects imbalance between personal and non-personal tweets |
| Random seed | 42 | Ensures reproducibility across runs |

### LSTM Model
The standard LSTM processes each tweet token-by-token in a **forward direction only**, maintaining a hidden state that accumulates contextual information. This is effective for tweets where health-mention signals typically appear in the subject-verb-object order (e.g., *'I took xanax for my anxiety'*).

### Bi-LSTM Model
The Bi-LSTM runs **two LSTM layers simultaneously** — one forward, one backward — and concatenates their outputs at each timestep. This gives the model awareness of both past and future context simultaneously, which is beneficial when key tokens appear at varying positions in the tweet (e.g., a health medication mentioned at the end that contextualises the beginning).

### Confusion Matrix Analysis
- **True Positives (TP):** Personal health tweets correctly classified — top priority for this task
- **True Negatives (TN):** Non-personal tweets correctly rejected
- **False Negatives (FN):** Missed personal health mentions — most costly error in health monitoring context
- **False Positives (FP):** Non-personal tweets incorrectly flagged as health mentions

### Key Observations
The dataset contains significant noise — many tweets mention medications in non-personal contexts (e.g., jokes, referring to others). Both models must learn to distinguish first-person health language from third-person references, which is the core challenge of this classification task. The Bi-LSTM's bidirectional context helps capture first-person pronouns (I, my, me) that may appear distant from the medication keyword in the sequence.